In [0]:
%run ./utils

# Imports

In [0]:
dbutils.widgets.text("silver_table_name", "tb_cliente")
silver_table_name = dbutils.widgets.get("silver_table_name")

df_cliente = load_df(silver_table_name)

display(df_cliente.limit(5))

# Atributos

In [0]:
# A dimensão guarda apenas atributos de negócio: a flag de e-mail corrigido e o timestamp de ingestão ficam na silver
df_atributos = df_cliente.select(
    F.col("id_cliente"),
    F.col("nome"),
    F.col("email"),
    F.col("regiao"),
    F.col("idade"),
    F.col("data_cadastro"),
    F.col("dia_vencimento_fatura")
)

# Chave

In [0]:
# A chave substituta isola a dimensão de mudanças no id de negócio e é criada na primeira coluna
df_dim_cliente = add_sequential_key(df_atributos, "cliente_sk")

# Verificação

In [0]:
# A dimensão deve ter uma linha por cliente e a chave deve ser única
total = df_dim_cliente.count()
cliente = df_dim_cliente.select("id_cliente").distinct().count()
chaves = df_dim_cliente.select("cliente_sk").distinct().count()
print(f"Linhas: {total} | id_cliente únicos: {cliente} | sk_cliente únicas: {chaves}")

# Salvamento

In [0]:
dbutils.widgets.text("gold_table_name", "dim_cliente")
gold_table_name = dbutils.widgets.get("gold_table_name")

In [0]:

save_df(df_dim_cliente, gold_table_name)